# Section 1: Why Passive RAG Breaks

*Duration: 15 minutes*

---

The Escalation Lab established a pattern: start with the simplest possible system, measure what it gets wrong, and only add complexity when the evidence justifies it.

That pattern produced a complete pipeline. A question arrives. The retriever pulls the top-k chunks from a vector store. The model reads those chunks and produces an answer. Every time. In that order.

If you worked through the Escalation Lab, you have seen this pipeline in action. If you are joining here directly, the pre-built outputs in `../prebuilt/` give you everything you need to run this section without having completed it first.

Either way, this section starts from the same place: a RAG pipeline that works reasonably well, and two questions it consistently gets wrong.

This section is not about improving retrieval or the model. It is about understanding what the architecture itself cannot do, and why that gap matters before we build anything new.

## 1.1 Re-Introducing the Baseline

The evaluation artifact below was produced at the end of the Escalation Lab. It captures the 10 evaluation questions and their final pass/fail classifications after RAG, Best-of-N, and fine-tuning were all applied.

If you completed the Escalation Lab and want to use your own results, point the path below at your generated file. Otherwise, the pre-built version is ready to go.

In [1]:
import json

EVAL_PATH = "../prebuilt/eval_results.json"  # swap path if using your own output

with open(EVAL_PATH, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

results = eval_data["results"]

passes   = [r for r in results if r["classification"] == "pass"]
failures = [r for r in results if r["classification"] != "pass"]

print(f"Total questions : {len(results)}")
print(f"Passes          : {len(passes)}")
print(f"Failures        : {len(failures)}")

Total questions : 10
Passes          : 6
Failures        : 4


You should see 8 passes and 2 remaining failures.

Those two questions survived every improvement the Escalation Lab applied: better chunking, RAG, Best-of-N sampling, and LoRA fine-tuning. They are not random noise. They are the pipeline telling you something specific about its own architecture.

If you are joining this lab without completing the Escalation Lab, here is what you need to know about these results:

- The corpus is the **Basic Fantasy RPG rulebook**, ingested and chunked using Docling
- The 10 questions cover character class abilities, combat rules, and equipment
- The pipeline uses cosine similarity retrieval against a ChromaDB vector store
- The model is `granite-3-2-8b-instruct` served via the Red Hat MaaS endpoint
- 8 of the 10 questions are answered correctly after all optimizations. 2 are not.

## 1.2 Looking at the Failures

Before categorizing anything, read the failures directly. Look at the retrieved chunks **before** reading the model's answer.

Ask yourself: if you were handed only those chunks and that question, could you produce the correct answer?

In [2]:
print("Remaining failures")
print("=" * 60)

for r in failures:
    print(f"\nQuestion : {r['question']}")
    print(f"Expected : {r['expected']}")
    print(f"Got      : {r['answer']}")
    print(f"\nChunks retrieved:")
    for i, chunk in enumerate(r.get("context_chunks", []), 1):
        print(f"  [{i}] {chunk[:150]}...")
    print("-" * 60)

Remaining failures

Question : Why can't Elves roll higher than a d6 for hit points?
Expected : Elves use a d6 for hit points because that is the hit die assigned to the Elf combination class in Basic Fantasy RPG.
Got      : According to the provided context, Elves never roll larger than six-sided dice (d6) for hit points. The reason for this restriction is not explicitly stated in the context.

Chunks retrieved:
------------------------------------------------------------

Question : What is the saving throw for a 3rd level Fighter against Dragon Breath?
Expected : Based on the Fighter saving throw table, a 3rd level Fighter has a Dragon Breath saving throw of 15.
Got      : The context does not provide specific information on the saving throw for a 3rd level Fighter against Dragon Breath. However, it does mention that characters may make a save vs. Dragon Breath for half damage. The exact saving throw (e.g., Strength, Dexterity, etc.) is not specified in the provided context.

Chunks

Read both outputs carefully before moving on.

In most cases, one of two things is happening:

- The chunks do not contain the information needed to answer the question
- The chunks contain the information, but answering correctly requires combining facts the model retrieved and then did not use

These are different problems. The pipeline cannot tell them apart. That is the architectural gap this lab addresses.

## 1.3 Naming the Three Failure Types

The failures you just read map to one of three categories. Naming them precisely matters because each one points to a different fix.

---

### Irrelevant Retrieval

The retriever returned chunks that are topically adjacent but do not contain the answer. The model received context that looked relevant and answered from it anyway. The answer is wrong because the input was wrong, and the pipeline had no way to detect that before committing to a response.

Run the cell below to inspect the cosine similarity scores for the failing questions. Values above 0.35 are worth examining. The retriever has no relevance threshold. It returns the closest chunks it has, relevant or not, and the model answers regardless.

In [4]:
pip install chromadb -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kfp 2.14.6 requires click==8.1.8, but you have click 8.3.2 which is incompatible.
odh-elyra 4.3.2 requires click==8.1.8, but you have click 8.3.2 which is incompatible.
opentelemetry-exporter-prometheus 0.60b1 requires opentelemetry-sdk~=1.39.1, but you have opentelemetry-sdk 1.40.0 which is incompatible.
ray 2.52.1 requires click!=8.3.*,>=7.0, but you have click 8.3.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [5]:
import chromadb

chroma_client = chromadb.PersistentClient(path="../prebuilt/chroma_db")
collection = chroma_client.get_collection("basic_fantasy_corpus")

print("Retrieval similarity scores for failing questions")
print("=" * 60)

for r in failures:
    results_raw = collection.query(
        query_texts=[r["question"]],
        n_results=3,
        include=["documents", "distances"]
    )
    print(f"\nQuestion: {r['question']}")
    for i, (doc, dist) in enumerate(
        zip(results_raw["documents"][0], results_raw["distances"][0]), 1
    ):
        flag = "  <-- examine this" if dist > 0.35 else ""
        print(f"  Chunk {i} | distance: {dist:.4f}{flag}")
        print(f"           | preview : {doc[:100]}...")

RuntimeError: [91mYour system has an unsupported version of sqlite3. Chroma                     requires sqlite3 >= 3.35.0.[0m
[94mPlease visit                     https://docs.trychroma.com/troubleshooting#sqlite to learn how                     to upgrade.[0m

---

### Implicit Reasoning

The answer exists in the corpus. The retriever found the right chunks. But the question requires combining two facts from different sections of the document, and the model did not make that connection.

This is not a retrieval failure. It is a reasoning gap that retrieval alone cannot close. Adding more training data will not fix it. Changing the chunking strategy will not fix it. The pipeline needs a different control structure.

---

### Out-of-Scope Questions

The answer does not exist anywhere in the corpus. The correct response is: *"I do not have that information."*

A passive pipeline does not produce that response. It retrieves whatever is closest and answers from it, producing a confident, grounded-sounding, incorrect answer.

---

Fill in the classification below based on your inspection of the outputs above. You will come back to this table in Section 2.

In [ ]:
# Complete this classification based on your inspection above.
# Failure types: "irrelevant_retrieval", "implicit_reasoning", "out_of_scope"

failure_classifications = [
    {
        "id": failures[0]["id"],
        "question": failures[0]["question"],
        "failure_type": "irrelevant_retrieval",  # <-- update this
        "evidence": ""                            # <-- one sentence explaining why
    },
    {
        "id": failures[1]["id"],
        "question": failures[1]["question"],
        "failure_type": "implicit_reasoning",     # <-- update this
        "evidence": ""                            # <-- one sentence explaining why
    }
]

print("Failure Classification")
print("=" * 60)
for fc in failure_classifications:
    print(f"\n  ID           : {fc['id']}")
    print(f"  Question     : {fc['question']}")
    print(f"  Failure type : {fc['failure_type']}")
    print(f"  Evidence     : {fc['evidence'] or '(not yet filled in)'}")

> **Facilitator note:** Participants often want to say "the model should just know better." Redirect that instinct. The failure type determines the fix. Irrelevant retrieval is a pipeline problem. Implicit reasoning requires a different control structure. Out-of-scope questions require the system to recognize when it cannot answer. None of those fixes are "train the model harder."

## 1.4 The Architectural Problem

A passive RAG pipeline is built around one assumption: retrieval will return something useful.

When that assumption holds, the pipeline works well. When it does not, the pipeline has no recovery path. It retrieved. It answered. It moved on.

This is not a flaw in the retriever or the model. It is a structural property of the architecture. The pipeline does not inspect the retrieval result before deciding whether to answer. It does not ask whether the retrieved context is relevant. It does not consider whether the question falls outside the scope of the corpus. It executes the same sequence every time.

Connect to the MaaS endpoint and run the passive pipeline on one of the failing questions so you can see the behavior directly.

In [ ]:
import os
from openai import OpenAI

api_key  = os.environ.get("MAAS_API_KEY")
base_url = os.environ.get("MAAS_BASE_URL")
model_id = os.environ.get("MAAS_MODEL_ID", "granite-3-2-8b-instruct")

client = OpenAI(api_key=api_key, base_url=base_url)

print(f"Connected to : {base_url}")
print(f"Model        : {model_id}")

In [ ]:
def passive_rag(question, collection, model_client, model_id, n_results=3):
    """
    The pipeline as built in the Escalation Lab.

    Step 1: Retrieve. Always.
    Step 2: Answer. Always.

    No inspection. No decision. No recovery path.
    """
    # Step 1: Retrieve
    retrieved = collection.query(
        query_texts=[question],
        n_results=n_results,
        include=["documents"]
    )
    context = "\n\n".join(retrieved["documents"][0])

    # Step 2: Answer
    prompt = f"""You are a rules assistant for Basic Fantasy RPG.
Use only the context below to answer the question.
If the context does not contain enough information, say so.

Context:
{context}

Question: {question}"""

    response = model_client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    return response.choices[0].message.content, context


question = failures[0]["question"]
answer, context_used = passive_rag(question, collection, client, model_id)

print(f"Question      : {question}")
print(f"Expected      : {failures[0]['expected']}")
print(f"\nContext given to model:")
print("-" * 40)
print(context_used[:400] + "...")
print("-" * 40)
print(f"\nModel answered: {answer}")

The pipeline ran. It retrieved. It answered.

It had no mechanism to detect that the retrieval was insufficient before producing that answer. The model was not malfunctioning. It did exactly what the architecture asked of it. The architecture asked the wrong thing.

That is the problem the agent loop solves. The loop gives the model a decision to make before it commits to an answer. The retriever does not go away. It becomes one option among several, and the model decides when to use it.

---

> **FIELD TAKEAWAY**
>
> A passive pipeline that always retrieves and always answers will always hallucinate on questions where retrieval fails. The fix is not a better model. It is a control structure that can inspect, decide, and choose a different path.

## Pre-Built Output

If the cells above did not execute due to endpoint availability or time constraints, run the cell below to load pre-built results and continue the discussion from there.

This is expected behavior during a workshop. Use the pre-built outputs without apology and keep the discussion moving.

In [ ]:
USE_PREBUILT = False  # Set to True if live execution was not available

if USE_PREBUILT:
    with open("../prebuilt/section1_outputs.json", "r", encoding="utf-8") as f:
        prebuilt = json.load(f)

    failures = prebuilt["failures"]
    failure_classifications = prebuilt["failure_classifications"]

    print("Loaded pre-built Section 1 outputs")
    print(f"Failures captured : {len(failures)}")
    print(f"Classifications   : {[f['failure_type'] for f in failure_classifications]}")
    print("\nReady to continue to Section 2.")
else:
    print("Using live results. Ready to continue to Section 2.")

---

## What Comes Next

Section 2 introduces the agent loop: the control structure that gives the model a decision to make before it commits to an answer.

The retriever does not go away. It becomes one option among several, and the model decides when to use it.

Move to `02_The_Agent_Loop.ipynb`.